# 01 – Exploratory Data Analysis
**Project:** War Economic Impact Predictor  
**Author:** Muhammad Farooq  
**Date:** 2026-03-03

## Objectives
1. Understand the shape, dtypes, and quality of the raw dataset  
2. Explore distributions of economic indicators across conflicts  
3. Identify correlations, outliers, and class imbalance  
4. Generate publication-quality visualizations saved to `reports/figures/`

---

In [ ]:
# ── Standard imports ──────────────────────────────────────────────────────────
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.io as pio

# ── Project root on path ──────────────────────────────────────────────────────
ROOT = Path().resolve().parent  # notebooks/ -> project root
sys.path.insert(0, str(ROOT))

from src.visualization.visualize import (
    plot_gdp_distribution,
    plot_numeric_distributions,
    plot_gdp_by_conflict_type,
    plot_gdp_by_region,
    plot_correlation_heatmap,
    plot_duration_vs_gdp,
    plotly_gdp_choropleth,
    plotly_scatter_matrix,
    plotly_inflation_boxplot,
)

# Inline plots
%matplotlib inline
plt.rcParams['figure.dpi'] = 120
pio.renderers.default = 'notebook'

print('Imports OK')

In [ ]:
# ── Load raw dataset ──────────────────────────────────────────────────────────
RAW_PATH = ROOT / 'data' / 'raw' / 'war_economic_impact_dataset.csv'
df = pd.read_csv(RAW_PATH, low_memory=False)

print(f'Shape: {df.shape}')
print(f'Columns ({len(df.columns)}): {df.columns.tolist()}')

In [ ]:
# ── Basic info ────────────────────────────────────────────────────────────────
df.head(5)

In [ ]:
df.dtypes

In [ ]:
df.describe(include='all').T

In [ ]:
# ── Missing value analysis ────────────────────────────────────────────────────
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df[missing_df['Missing Count'] > 0]

In [ ]:
# ── Duplicates ────────────────────────────────────────────────────────────────
n_dup = df.duplicated().sum()
print(f'Duplicate rows: {n_dup:,} ({n_dup/len(df)*100:.2f}%)')

## 2. Univariate Analysis

In [ ]:
# ── Target variable distribution ──────────────────────────────────────────────
fig = plot_gdp_distribution(df, save=True)
plt.show()

print('Skewness:', df['GDP_Change_%'].skew().round(3))
print('Kurtosis:', df['GDP_Change_%'].kurt().round(3))

In [ ]:
fig = plot_numeric_distributions(df, save=True)
plt.show()

In [ ]:
# ── Categorical value counts ───────────────────────────────────────────────────
for col in ['Conflict_Type', 'Region', 'Status', 'Black_Market_Activity_Level',
            'Most_Affected_Sector', 'War_Profiteering_Documented']:
    print(f'\n{col}:')
    print(df[col].value_counts().to_string())

## 3. Bivariate Analysis

In [ ]:
fig = plot_gdp_by_conflict_type(df, save=True)
plt.show()

In [ ]:
fig = plot_gdp_by_region(df, save=True)
plt.show()

In [ ]:
fig = plotly_inflation_boxplot(df)
fig.show()

In [ ]:
fig = plotly_gdp_choropleth(df)
fig.show()

## 4. Correlation Analysis

In [ ]:
fig = plot_correlation_heatmap(df, top_n=18, save=True)
plt.show()

In [ ]:
# ── Pearson correlations with target ─────────────────────────────────────────
num_df = df.select_dtypes(include=[np.number])
target_corr = num_df.corr()['GDP_Change_%'].drop('GDP_Change_%').sort_values()

fig, ax = plt.subplots(figsize=(8, 10))
target_corr.plot(kind='barh', color=target_corr.map(lambda v: 'steelblue' if v < 0 else 'coral'), ax=ax)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Pearson Correlation with GDP Change (%)')
plt.tight_layout()
plt.show()

In [ ]:
fig = plotly_scatter_matrix(df)
fig.show()

## 5. Outlier Detection

In [ ]:
# ── IQR-based outlier counting ────────────────────────────────────────────────
outlier_cols = [
    'GDP_Change_%', 'Inflation_Rate_%', 'Currency_Devaluation_%',
    'Cost_of_War_USD', 'Estimated_Reconstruction_Cost_USD',
    'Currency_Black_Market_Rate_Gap_%',
]
outlier_cols = [c for c in outlier_cols if c in df.columns]

for col in outlier_cols:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    n_out = ((df[col] < q1 - 1.5*iqr) | (df[col] > q3 + 1.5*iqr)).sum()
    print(f'{col:45s}: {n_out:5,} outliers ({n_out/len(df)*100:.2f}%)')

In [ ]:
# ── Duration analysis  ────────────────────────────────────────────────────────
df2 = df.copy()
df2['Conflict_Duration_Years'] = (df2['End_Year'] - df2['Start_Year']).clip(lower=0)

# Add Severity_Label for colouring
conditions = [
    df2['GDP_Change_%'] > -10,
    (df2['GDP_Change_%'] <= -10) & (df2['GDP_Change_%'] > -25),
    (df2['GDP_Change_%'] <= -25) & (df2['GDP_Change_%'] > -50),
    df2['GDP_Change_%'] <= -50,
]
df2['Severity_Label'] = [0,1,2,3]
import numpy as np
df2['Severity_Label'] = np.select(conditions, [0, 1, 2, 3], default=0)

fig = plot_duration_vs_gdp(df2, save=True)
plt.show()

## 6. EDA Summary & Key Findings

| Finding | Detail |
|---------|--------|
| Dataset size | 100,001 rows × 28 columns |
| No missing values | Dataset is fully populated |
| Target skew | GDP_Change_% is left-skewed (mean ≈ −25 to −30%) |
| Highest inflation conflicts | Asymmetric wars & civil wars |
| Strongest corr w/ GDP | Unemployment spike, poverty rate, currency devaluation |
| Class imbalance | Catastrophic class (~5–10%) — will need stratified split |

> **Next step:** `02_feature_engineering.ipynb`